# M4 based on RQ1_fixed

In [ ]:
import sys
import os
import csv
import random
import shutil
from datetime import datetime
import torch

parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from program_classifier import ProgramClassifier, _natural_key

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(2137)

How well can RoBERTa identify three Gang of Four patterns (Factory Method, Prototype, Singleton) in Java source code across 30 examples (15 true, 15 false) each?

In [ ]:
# Where the frozen RQ1 samples are written (relative to this notebooks/ folder)
RQ1_ROOT = "../datasets/rq1_datasets"

# Positives are the keyword-named copies in <Pattern>/name/ (unambiguous IDs);
# negatives come from each pattern's non-* folder.
RQ1_SOURCES = {
    "factory_method": {
        "true":  "../datasets/Factory method/name",             # 11 *.java
        "false": "../datasets/Factory method/nonfactorymethod", # 82 *.java
    },
    "prototype": {
        "true":  "../datasets/Prototype/name",                  # 32 *.java
        "false": "../datasets/Prototype/nonprototype",          # 61 *.java
    },
    "singleton": {
        "true":  "../datasets/Singleton/name",                  # 25 *.java
        "false": "../datasets/Singleton/nonsingleton",          # 68 *.java
    },
}

def build_rq1_dataset(pattern, n_true=15, n_false=None, seed=None):
    """Copy one frozen RQ1 sample into ../datasets/rq1_datasets/<pattern>/.

    pattern : "factory_method" | "prototype" | "singleton"
    n_true  : number of TRUE examples (capped at what the dataset provides)
    n_false : number of FALSE examples; None -> same as the actual n_true (1:1)
    seed    : None -> first N files in stable (natural) sorted order
              int  -> reproducible random sample using that seed

    Creates <pattern>/pattern/ (label 1) and <pattern>/non-pattern/ (label 0),
    rebuilds both folders on every call, and writes <pattern>/selection.csv
    with the frozen file IDs.
    """
    if pattern not in RQ1_SOURCES:
        raise ValueError(f"Unknown pattern {pattern!r}; choose from {list(RQ1_SOURCES)}")

    src_true  = RQ1_SOURCES[pattern]["true"]
    src_false = RQ1_SOURCES[pattern]["false"]

    true_files  = sorted((f for f in os.listdir(src_true)  if f.endswith(".java")), key=_natural_key)
    false_files = sorted((f for f in os.listdir(src_false) if f.endswith(".java")), key=_natural_key)

    # "max per class": never copy more than the dataset actually has
    n_true_sel  = min(n_true, len(true_files))
    n_false_sel = min(n_true_sel if n_false is None else n_false, len(false_files))

    if n_true_sel < n_true:
        print(f"  note: {pattern} has only {len(true_files)} true examples; using all of them.")
    if n_false is not None and n_false_sel < n_false:
        print(f"  note: {pattern} has only {len(false_files)} false examples; using all of them.")

    # pick the sample
    if seed is None:
        sel_true, sel_false = true_files[:n_true_sel], false_files[:n_false_sel]
        rule = "first N files in stable sorted order"
    else:
        rng = random.Random(seed)
        sel_true  = rng.sample(true_files,  n_true_sel)
        sel_false = rng.sample(false_files, n_false_sel)
        rule = f"random sample, seed={seed}"

    # rebuild the destination so it contains exactly this selection
    target    = os.path.join(RQ1_ROOT, pattern)
    dst_true  = os.path.join(target, "pattern")
    dst_false = os.path.join(target, "non-pattern")
    if os.path.exists(target):
        shutil.rmtree(target)
    os.makedirs(dst_true)
    os.makedirs(dst_false)

    for f in sel_true:
        shutil.copy2(os.path.join(src_true, f), os.path.join(dst_true, f))
    for f in sel_false:
        shutil.copy2(os.path.join(src_false, f), os.path.join(dst_false, f))

    # freeze the choice on disk (evidence for the report)
    with open(os.path.join(target, "selection.csv"), "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["class", "label", "filename", "source_folder"])
        writer.writerows([["pattern", 1, f, src_true] for f in sel_true])
        writer.writerows([["non-pattern", 0, f, src_false] for f in sel_false])
        writer.writerow([])
        writer.writerow([f"selection rule: {rule}"])
        writer.writerow([f"created: {datetime.now().isoformat(timespec='seconds')}"])

    print(f"{pattern}: {n_true_sel} true + {n_false_sel} false -> {target}")
    print(f'  java_programs_folder = "{target}"')
    print(f'  class_folders = {{"non-pattern": 0, "pattern": 1}}')

    return {"pattern": pattern, "n_true": n_true_sel, "n_false": n_false_sel, "folder": target}


def build_all_rq1(n_true=15, n_false=None, seed=None):
    """Build all three RQ1 patterns with the same true/false settings."""
    results = [build_rq1_dataset(p, n_true=n_true, n_false=n_false, seed=seed)
               for p in RQ1_SOURCES]
    print("\nSummary")
    for r in results:
        print(f"  {r['pattern']:15s} {r['n_true']:2d} true + {r['n_false']:2d} false "
              f"({r['n_true'] + r['n_false']:2d} files) -> {r['folder']}")
    return results

In [ ]:
build_all_rq1(n_true=15, n_false=None)

# Factory Method Results

Expected Results:

Precision: 0.87
Recall: 0.77
F1-score: 0.78

Actual Results (stratified 5-fold CV, weighted average):

Precision: 0.90
Recall: 0.90
F1-score: 0.90

Sample: 11 true + 10 false (21 files; `nonfm (11).java` removed by FIX 3)

In [ ]:
# Define the path to the folder containing your Java programs
java_programs_path = "../datasets/rq1_datasets/" 

design_patterns = ["factory_method", "prototype", "singleton"]
all_results = {}

for pattern in design_patterns:
    print(f"\n\n---------------------------- PROCESSING PATTERN: {pattern.upper()} ----------------------------")
    java_programs_folder = os.path.join(java_programs_path, pattern)
    classifier = ProgramClassifier(java_programs_folder=java_programs_folder, distance_metric='euclidean', device=device)
    results = classifier.run_classification()
    classifier.print_results(results)
    all_results[pattern] = results